# Archiver

The Archiver class is responsible to insert the model predictions into an xarray dataset at the correct coordinates and save the result to disk.
To do this, we use the `crest.archiver.Archiver` class. By passing in the model predictions from the predict() method of the `crest.model.Model` class, we can start writing the predictions to disk.

Here, we are going to demonstrate that how the Archiver works. But, before going into that part, let's first load a toy dataset for a soil moisture prediction model:

In [ ]:
 # Ensure we can import crest
%cd -q ..

# auto reload 
%load_ext autoreload
%autoreload 2

In [ ]:
import tlz
from pathlib import Path
from tqdm import tqdm
import numpy as np
import xarray as xr
import tensorflow as tf

from crest.data_server.DataServer import DataServer
from crest.data.loading import Datafile, Dataset
from crest.data import Batcher
from crest.archiver import Archiver
from crest.model.HierarchalTensorGraph import HierarchalTensorGraph as HTG
from crest.model.Model import Model
from crest.utils.batcher_for_archiver import batch_for_archiver

In [ ]:
 # For reproducibility
tf.keras.utils.set_random_seed(812)
tf.config.experimental.enable_op_determinism()

In [ ]:
# Copy data to local directory from data server
ROOT_PATH = Path(DataServer.load('soil_moisture',''))

#### Define the required variables for the Dataset

In [ ]:
# Define data locations
locations = {
    'ERA5'       : ROOT_PATH.joinpath('ERA5.zarr'),
    'SMAP'       : ROOT_PATH.joinpath('SMAP.zarr'),
    'Soil'       : ROOT_PATH.joinpath('StaticAttributes.zarr/Soil'),
    'Irrigation' : ROOT_PATH.joinpath('StaticAttributes.zarr/Irrigation'),
    
}

# Define data attributes
features = {
    'ERA5'       : ['cape', 'cp', 'csfr','cvh', 'cvl', 'e', 'es',
                    'fal','lai_hv', 'lai_lv', 'msdwlwrf', 'msdwswrf',
                    'pev', 'sd','skt','smlt','sp', 'sro','slhf',
                    'sshf','ssro','stl1', 'stl2', 'stl3', 'stl4',
                    't2m','tp','u10', 'v10', 'z'],
    'Irrigation' : ['Actual', 'Equipped', 'GroundWater', 
                    'Non-Conventional', 'SurfaceWater'],
    'Soil'       : ['Clay_frac', 'Sand_frac', 'Silt_frac'],
    'SMAP'       : ['soil_moisture']
}

# Define depth and temporal extents
depth = {
    'ERA5': {'datetime': (335,0), 'latitude': 0, 'longitude': 0},
    'Irrigation': {'latitude': 0, 'longitude': 0},
    'Soil': {'latitude': 0, 'longitude': 0},
    'SMAP': {'datetime': 0, 'latitude': 0, 'longitude':0},
    }

extent_train = {'SMAP':{'datetime': [np.datetime64('2015-05-14 00:00:00'), np.datetime64('2015-05-29 23:00:01')]},
                'ERA5': {},
                'Soil': {},
                'Irrigation': {}
                }


#### Create the dataset and data batcher

In [ ]:
# Create Datasets and Batchers
dataset_archive = Dataset([
                Datafile(
                    location     = locations[source],
                    features     = features[source],
                    window_depth = depth[source],
                    extent=extent_train[source],
                ) for source in features])

batch_train = Batcher(dataset_archive, **{
    'batch_size' : 10,
    'features'   : None,
    'repeat'     : False,
    'shuffle'    : True,
    'numblocks'  : [1,1,1],
    'workers'    : 1,
})

#### Create a soil moisture model

In [ ]:
# Soil moisture model
class sm_model(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()
        self._temporal = tf.keras.layers.LSTM(units=256, name='temporal')
        self._head = tf.keras.layers.Dense(1)

    def __call__(self, X):
    
        era5 = tf.stack([X[k] for k in features['ERA5']], axis=-1)
        era5 = tf.keras.layers.Lambda(lambda x: tf.squeeze(x, axis=[2,3]))(era5)
        x = self._temporal(era5)
        soil = tf.stack([X[k] for k in features['Soil']], axis=-1)
        soil = tf.squeeze(tf.cast(soil, dtype=tf.float32), axis=[1,2])
        irrigation = tf.stack([X[k] for k in features['Irrigation']], axis=-1)
        irrigation = tf.squeeze(tf.cast(irrigation, dtype=tf.float32), axis=[1,2])
        x = tf.concat([x, soil, irrigation], axis=-1)
    
        output = {'soil_moisture': self._head(x)}
    
        return output
    
# Create HTG basenode
inputs = {f: tf.TensorSpec((None,336, 1, 1)) 
          if f in features['ERA5'] else tf.TensorSpec((None,1, 1)) for f in features['ERA5'] + features['Soil'] + features['Irrigation']}

graph = HTG(
    node=sm_model(),
    name='SM',
    inputs=inputs,
    outputs = {f: tf.TensorSpec((None,1)) for f in features['SMAP']}
)

# Create model
model = Model(graph)

# Compile model
model.compile(**{
    'optimizer' : 'Adam', 
    'loss'      : 'mse', 
    'metrics'   : [tf.keras.metrics.RootMeanSquaredError()]
})


#### Predict the model outputs for the input dataset and archive them using the Archiver class. 

We must select a data_schema for the Archiver. data_schema is used to define the coordinates of the archived xarray dataset. It can be either one of the Datafiles of the crest dataset or a dictionary of coordinate keys and its values. If the crest dataset is used, the Archiver class considers the finest resolution of the Datafiles for each coordinate unless a datafile_index is specified. Here we chose the SMAP Datafile as the schema by datafile_index:

In [ ]:
with Archiver(output_path = 'model_predictions.zarr',
                              data_schema = dataset_archive,
                              datafile_index='smap',
                              overwrite=True,) as archiver:

    # start getting the samples
    with batch_train as batcher:
        # this function makes the batch ready for archiving
        # by getting the asked input variables and coordinate
        # info
        batcher_archiver = batch_for_archiver(batcher,
                                              model_inputs = list(inputs.keys()),
                                              coordinate_df = 'smap',
                                              dataset=dataset_archive,)
        for coord, value in tqdm(batcher_archiver):

            pred = model.predict_on_batch(value)
            archiver.archive(coord, pred)

#### Load the archived data

In [ ]:
output_data = xr.open_zarr('model_predictions.zarr')
output_data['soil_moisture']